# Tani-Awas: Sistem Prediksi Dini Risiko Kekeringan Pertanian

**Dataset:** 144 Observasi Pemantauan Spasial-Multitemporal (12 Kecamatan Sentra Pangan, 12 Minggu Pemantauan).  
**Objective:** Mengintegrasikan data penginderaan jauh indeks vegetasi satelit (Sentinel-2 / MODIS NDVI via Google Earth Engine), data akumulasi curah hujan harian/mingguan BMKG, dan data suhu permukaan lahan (LST) untuk memprediksi risiko kekeringan pertanian 2–4 minggu ke depan serta mengonversi dampaknya terhadap potensi kehilangan hasil panen (Ton) dan kerugian finansial (Juta Rupiah).

### Tahapan Pemodelan:
1. **Prapemrosesan Data**: Penyelarasan resolusi spasial dan multitemporal data NDVI, curah hujan BMKG, dan LST.
2. **Formulasi Indeks Risiko Kekeringan**: Menghitung *Drought Risk Score* multivariat dan status peringatan dini.
3. **Analisis Eksploratif & Visualisasi 300 DPI**: Visualisasi terpisah A, B, C, dan D (NDVI vs Skor Risiko, Akumulasi Kerugian per Kecamatan, Heatmap Temporal Risiko, dan Densitas Curah Hujan BMKG).
4. **Pemodelan Ekonometrika & Regresi OLS**: Evaluasi hubungan empiris antara defisit air terhadap penurunan biomassa tanaman.
5. **Rekomendasi Taktis Petani & Dinas Pertanian**: Protokol mitigasi mingguan adaptif.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols

os.makedirs('images', exist_ok=True)
os.makedirs('data', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
    'figure.dpi': 300,
    'savefig.dpi': 300
})

print('Environment setup complete. Output directory ready.')

In [ ]:
df = pd.read_csv('data/agridrought_monitoring_data.csv')
print('--- Dataset Shape ---')
print(df.shape)
print('\n--- Head Data ---')
print(df.head())

In [ ]:
# Model Regresi OLS Penentu Skor Risiko Kekeringan
model = ols('drought_risk_score ~ ndvi_sentinel2 + curah_hujan_bmkg_mm + suhu_permukaan_lst_c', data=df).fit()
print(model.summary())

In [ ]:
# 1. Indeks Vegetasi Satelit (NDVI) vs Skor Risiko Kekeringan
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    df['ndvi_sentinel2'],
    df['drought_risk_score'],
    s=df['curah_hujan_bmkg_mm'] * 3,
    c=df['potensi_kerugian_juta_rp'],
    cmap='YlOrRd',
    alpha=0.8,
    edgecolors='black',
    linewidth=0.5
)
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Estimasi Kerugian Finansial (Juta Rp)', fontsize=9)
ax.set_title('Indeks Vegetasi Satelit (NDVI) vs Skor Risiko Kekeringan Pertanian', fontweight='bold', pad=12)
ax.set_xlabel('Nilai NDVI Sentinel-2 (0.0 = Kering Ekstrem, 1.0 = Sehat)')
ax.set_ylabel('Skor Risiko Kekeringan (0.0 - 1.0)')
ax.axhline(0.70, color='#dc2626', linestyle='--', linewidth=1.2, label='Ambang Batas Bahaya (Tinggi >= 0.70)')
ax.axhline(0.45, color='#eab308', linestyle=':', linewidth=1.2, label='Ambang Batas Waspada (Sedang >= 0.45)')
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('images/ndvi_vs_drought_risk.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Akumulasi Proyeksi Kerugian Finansial per Wilayah Kecamatan
fig, ax = plt.subplots(figsize=(9, 6))
loss_summary = df.groupby('kecamatan')['potensi_kerugian_juta_rp'].sum().sort_values(ascending=True)
bars = ax.barh(loss_summary.index, loss_summary.values, color='#1e3a8a', height=0.6, edgecolor='black', linewidth=0.5)
ax.set_title('Akumulasi Proyeksi Kerugian Finansial per Wilayah Kecamatan', fontweight='bold', pad=12)
ax.set_xlabel('Total Estimasi Kerugian (Juta Rupiah)')
ax.grid(axis='y')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 80, bar.get_y() + bar.get_height()/2, f'Rp{width:,.1f}M', ha='left', va='center', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.savefig('images/financial_loss_by_kecamatan.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Evolusi Skor Risiko Kekeringan Lahan Pertanian (Kecamatan vs Minggu Pemantauan)
fig, ax = plt.subplots(figsize=(10, 6))
pivot_heatmap = df.pivot_table(index='kecamatan', columns='week', values='drought_risk_score', aggfunc='mean')
sns.heatmap(pivot_heatmap, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, cbar=True, linewidths=0.5)
ax.set_title('Evolusi Skor Risiko Kekeringan Lahan Pertanian (Kecamatan vs Minggu Pemantauan)', fontweight='bold', pad=12)
ax.set_xlabel('Minggu Pemantauan (Week 1 - 12)')
ax.set_ylabel('Kecamatan')
plt.tight_layout()
plt.savefig('images/temporal_drought_risk_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Distribusi Curah Hujan BMKG Berdasarkan Status Tingkat Risiko Kekeringan
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(df[df['status_risiko'] == 'RENDAH (HIJAU)']['curah_hujan_bmkg_mm'], fill=True, label='Status Rendah (Hijau)', ax=ax, color='#16a34a', alpha=0.3)
sns.kdeplot(df[df['status_risiko'] == 'SEDANG (KUNING)']['curah_hujan_bmkg_mm'], fill=True, label='Status Waspada (Kuning)', ax=ax, color='#ca8a04', alpha=0.3)
sns.kdeplot(df[df['status_risiko'] == 'TINGGI (MERAH)']['curah_hujan_bmkg_mm'], fill=True, label='Status Bahaya (Merah)', ax=ax, color='#dc2626', alpha=0.3)
ax.set_title('Distribusi Curah Hujan BMKG Berdasarkan Status Tingkat Risiko Kekeringan', fontweight='bold', pad=12)
ax.set_xlabel('Akumulasi Curah Hujan Mingguan BMKG (mm)')
ax.set_ylabel('Fungsi Densitas Probabilitas (PDF)')
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig('images/rainfall_deficit_density.png', dpi=300, bbox_inches='tight')
plt.show()